# Two further experiments

**A. What actually causes the SMOTE inflation?** Pre-split SMOTE changes two things at once: fake rows leak into training, and the test fold becomes half synthetic at 50% prevalence. This scores the same leaky model **on the real encounters only**, which isolates contamination from the change in evaluation target.

**B. Does leakage manufacture apparent value for a useless feature?** The enrichment proxies carry no information beyond variables the model already has, and under correct evaluation they change nothing. This tests whether they *appear* to help under the leaky protocol.

Experiment B needs your enriched dataset, which is in your Google Drive. Cell 2 mounts Drive and finds it automatically.

**Before you run:** Runtime → Change runtime type → **T4 GPU**. Then Run all. About 12 minutes.

In [ ]:
#@title 1. Environment
!pip install -q xgboost==3.2.0 scikit-learn==1.8.0 imbalanced-learn==0.14.1 ucimlrepo 2>/dev/null
import numpy as np, pandas as pd, sklearn, xgboost as xgb, warnings, ssl, urllib.request, os, glob
warnings.filterwarnings('ignore')
try:
    d0=xgb.DMatrix(np.zeros((16,3)),label=np.array([0,1]*8))
    xgb.train({'tree_method':'hist','device':'cuda'},d0,num_boost_round=1); USE_GPU=True
except Exception: USE_GPU=False
print('sklearn',sklearn.__version__,'| xgboost',xgb.__version__,'| GPU',USE_GPU)

from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

TARGET, GROUP, N_SPLITS = 'readmitted','patient_nbr',5
CAT_AS_STR=['admission_type_id','discharge_disposition_id','admission_source_id','race']
SEED=42

def mk(pw,seed=SEED):
    kw=dict(n_estimators=500,max_depth=4,learning_rate=0.03,subsample=0.8,colsample_bytree=0.7,
            min_child_weight=5,reg_lambda=2.0,scale_pos_weight=pw,eval_metric='logloss',
            random_state=seed,tree_method='hist')
    if USE_GPU: kw['device']='cuda'
    return XGBClassifier(**kw)

def encode(df):
    if df[TARGET].dtype==object: df[TARGET]=(df[TARGET].astype(str)=='<30').astype(int)
    y=df[TARGET].astype(int).values; g=df[GROUP].values
    X=df.drop(columns=[TARGET,GROUP])
    for c in CAT_AS_STR:
        if c in X.columns: X[c]=X[c].astype(str)
    cat=[c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    return pd.get_dummies(X,columns=cat,dummy_na=False).astype(float).values.astype(float),y,g

In [ ]:
#@title 2. Mount Drive and locate the enriched dataset
from google.colab import drive
drive.mount('/content/drive')

hits=glob.glob('/content/drive/**/uci_mean_clinical.csv',recursive=True) \
   + glob.glob('/content/drive/**/uci_mean_enrichment.csv',recursive=True)
base_hits=glob.glob('/content/drive/**/uci_baseline.csv',recursive=True)
print('enriched candidates:'); [print('  ',h) for h in hits]
print('baseline candidates:'); [print('  ',h) for h in base_hits]
ENRICHED = hits[0] if hits else None
print('\nusing enriched:', ENRICHED)
if ENRICHED is None:
    print('>>> Not found. Experiment B will be skipped; Experiment A still runs.')

In [ ]:
#@title 3. Rebuild the baseline cohort from the public UCI dataset
ctx=ssl.create_default_context(); ctx.check_hostname=False; ctx.verify_mode=ssl.CERT_NONE
_o=urllib.request.urlopen
urllib.request.urlopen=lambda *a,**k:_o(*a,context=ctx,**{kk:vv for kk,vv in k.items() if kk!='context'})
from ucimlrepo import fetch_ucirepo
d=fetch_ucirepo(id=296)
raw=pd.concat([p for p in [d.data.ids,d.data.features,d.data.targets] if p is not None],axis=1)
EXPIRED={11,13,14,19,20,21}
DR={'has_diabetes_dx':[(250,250.99)],'has_circulatory_dx':[(390,459)],'has_respiratory_dx':[(460,519)],
    'has_renal_dx':[(580,629)],'has_digestive_dx':[(520,579)],'has_infectious_dx':[(1,139)],
    'has_injury_dx':[(800,999)],'has_neoplasm_dx':[(140,239)],'has_symptoms_dx':[(780,799)]}
u=raw.replace('?',np.nan).copy(); u=u[~u['discharge_disposition_id'].isin(EXPIRED)].copy()
for c in ['diag_1','diag_2','diag_3']:
    u[c]=pd.to_numeric(u[c].astype(str).str.replace('V|E','10',regex=True),errors='coerce')
for dis,rg in DR.items():
    m=False
    for lo,hi in rg: m=m|u[['diag_1','diag_2','diag_3']].apply(lambda col:col.between(lo,hi)).any(axis=1)
    u[dis]=m.astype(int)
MED=['metformin','repaglinide','nateglinide','chlorpropamide','glimepiride','acetohexamide','glipizide',
 'glyburide','tolbutamide','pioglitazone','rosiglitazone','acarbose','miglitol','troglitazone','tolazamide',
 'examide','citoglipton','insulin','glyburide-metformin','glipizide-metformin','glimepiride-pioglitazone',
 'metformin-rosiglitazone','metformin-pioglitazone']
med=[c for c in MED if c in u.columns]
u['med_change_count']=u[med].isin(['Up','Down']).sum(axis=1)
u['comorbidity_count']=u[list(DR)].sum(axis=1)
u['total_prior_visits']=(u['number_inpatient'].fillna(0)+u['number_emergency'].fillna(0)+u['number_outpatient'].fillna(0))
u=u.drop(columns=['diag_1','diag_2','diag_3'])
amap={'[0-10)':None,'[10-20)':None,'[20-30)':'20-39','[30-40)':'20-39','[40-50)':'40-59','[50-60)':'40-59',
 '[60-70)':'>=60','[70-80)':'>=60','[80-90)':'>=60','[90-100)':'>=60'}
u['age']=u['age'].map(amap); u=u.dropna(subset=['age'])
u=u[u['gender'].isin(['Male','Female'])]
u['gender']=u['gender'].map({'Male':1,'Female':2}).astype(int)
u['race']=u['race'].map({'Caucasian':3,'AfricanAmerican':4,'Hispanic':2,'Asian':5,'Other':5}).fillna(5).astype(int)
u['readmitted']=(u['readmitted'].astype(str)=='<30').astype(int)
u=u.drop(columns=[c for c in ['encounter_id','weight','payer_code','medical_specialty'] if c in u.columns])
u['patient_nbr']=u['patient_nbr'].astype(int)
base=u.drop(columns=['on_insulin'],errors='ignore').reset_index(drop=True)
assert (len(base),base.patient_nbr.nunique(),int(base.readmitted.sum()))==(98490,69311,11271)
print('cohort matches:',len(base),base.patient_nbr.nunique(),int(base.readmitted.sum()))
Xb,yb,gb=encode(base.copy()); print('baseline predictors:',Xb.shape[1])

## A. Contamination vs changed evaluation target

In [ ]:
#@title 4. Score the leaky model on real encounters only
def presplit_smote_oof(Xv,y,g,grouped=True,seed=SEED):
    """Returns (auroc_on_all_rows, auroc_on_real_rows_only).
       SMOTE appends synthetic rows after the originals, so the first len(y)
       rows of the resampled matrix are exactly the real encounters."""
    Xr,yr=SMOTE(random_state=seed).fit_resample(Xv,y)
    n_real=len(y)
    gr=np.concatenate([g,np.arange(g.max()+1,g.max()+1+(len(yr)-n_real))])
    if grouped:
        splits=StratifiedGroupKFold(N_SPLITS,shuffle=True,random_state=seed).split(np.zeros(len(yr)),yr,gr)
    else:
        splits=StratifiedKFold(N_SPLITS,shuffle=True,random_state=seed).split(np.zeros(len(yr)),yr)
    oof=np.zeros(len(yr))
    for tr,te in splits:
        sc=StandardScaler().fit(Xr[tr]); m=mk(1.0,seed); m.fit(sc.transform(Xr[tr]),yr[tr])
        oof[te]=m.predict_proba(sc.transform(Xr[te]))[:,1]
    auc_all =roc_auc_score(yr,oof)
    auc_real=roc_auc_score(yr[:n_real],oof[:n_real])   # original encounters only
    return auc_all,auc_real,yr[:n_real].mean()

# correct protocol reference in this environment
pw=(yb==0).sum()/max((yb==1).sum(),1)
oof_D=np.zeros(len(yb))
for tr,te in StratifiedGroupKFold(N_SPLITS,shuffle=True,random_state=SEED).split(np.zeros(len(yb)),yb,gb):
    sc=StandardScaler().fit(Xb[tr]); m=mk(pw); m.fit(sc.transform(Xb[tr]),yb[tr])
    oof_D[te]=m.predict_proba(sc.transform(Xb[te]))[:,1]
auc_correct=roc_auc_score(yb,oof_D)
print(f'[D] correct protocol                         AUROC={auc_correct:.4f}')

a_all,a_real,prev_real=presplit_smote_oof(Xb,yb,gb,grouped=True)
print(f'[A] pre-split SMOTE, scored on ALL rows      AUROC={a_all:.4f}   (50% prevalence, half synthetic)')
print(f'[A] pre-split SMOTE, scored on REAL rows only AUROC={a_real:.4f}  (prevalence {prev_real:.3f})')
print()
print(f'total inflation      {a_all-auc_correct:+.4f}')
print(f'  contamination only {a_real-auc_correct:+.4f}   <- leakage into training, judged on real patients')
print(f'  synthetic test set {a_all-a_real:+.4f}   <- remainder, from scoring on synthetic rows')

## B. Does leakage manufacture apparent value for a redundant feature?

In [ ]:
#@title 5. Enrichment under the correct protocol and under the leaky protocol
if ENRICHED is None:
    print('Enriched file not found in Drive - skipping Experiment B.')
else:
    enr=pd.read_csv(ENRICHED, low_memory=False)
    # align to the same cohort definition
    if enr[TARGET].dtype==object: enr[TARGET]=(enr[TARGET].astype(str)=='<30').astype(int)
    print('enriched rows:',len(enr),'| events:',int(enr[TARGET].sum()))
    Xe,ye,ge=encode(enr.copy())
    print('enriched predictors:',Xe.shape[1],'(baseline was',Xb.shape[1],')')

    # --- correct protocol, enriched ---
    pwe=(ye==0).sum()/max((ye==1).sum(),1)
    oof_e=np.zeros(len(ye))
    for tr,te in StratifiedGroupKFold(N_SPLITS,shuffle=True,random_state=SEED).split(np.zeros(len(ye)),ye,ge):
        sc=StandardScaler().fit(Xe[tr]); m=mk(pwe); m.fit(sc.transform(Xe[tr]),ye[tr])
        oof_e[te]=m.predict_proba(sc.transform(Xe[te]))[:,1]
    auc_e_correct=roc_auc_score(ye,oof_e)

    # --- leaky protocol, baseline and enriched ---
    a_base_leaky,_,_ = presplit_smote_oof(Xb,yb,gb,grouped=False)
    a_enr_leaky ,_,_ = presplit_smote_oof(Xe,ye,ge,grouped=False)

    print()
    print(f"{'protocol':34s} {'baseline':>10s} {'enriched':>10s} {'difference':>12s}")
    print(f"{'correct (patient-grouped, no SMOTE)':34s} {auc_correct:10.4f} {auc_e_correct:10.4f} {auc_e_correct-auc_correct:+12.4f}")
    print(f"{'leaky (pre-split SMOTE + enc folds)':34s} {a_base_leaky:10.4f} {a_enr_leaky:10.4f} {a_enr_leaky-a_base_leaky:+12.4f}")
    print()
    print('If the enrichment difference is ~0 under the correct protocol but positive under')
    print('the leaky one, leakage manufactures apparent value for a feature that carries no')
    print('information beyond variables the model already has.')

In [ ]:
#@title 6. Summary
lines=[]
def log(s): print(s); lines.append(s)
log(f'environment: xgboost {xgb.__version__}, sklearn {sklearn.__version__}, GPU {USE_GPU}')
log('')
log('A. DECOMPOSING THE SMOTE INFLATION')
log(f'  correct protocol                          {auc_correct:.4f}')
log(f'  pre-split SMOTE, all rows scored          {a_all:.4f}  ({a_all-auc_correct:+.4f})')
log(f'  pre-split SMOTE, real encounters only     {a_real:.4f}  ({a_real-auc_correct:+.4f})')
log(f'  share of inflation from contamination     {100*(a_real-auc_correct)/(a_all-auc_correct):.1f}%')
log(f'  share from scoring on synthetic rows      {100*(a_all-a_real)/(a_all-auc_correct):.1f}%')
log('')
if ENRICHED is not None:
    log('B. ENRICHMENT UNDER EACH PROTOCOL')
    log(f'  correct: baseline {auc_correct:.4f} vs enriched {auc_e_correct:.4f} ({auc_e_correct-auc_correct:+.4f})')
    log(f'  leaky:   baseline {a_base_leaky:.4f} vs enriched {a_enr_leaky:.4f} ({a_enr_leaky-a_base_leaky:+.4f})')
open('leakage_mechanism.txt','w').write('\n'.join(lines))
from google.colab import files; files.download('leakage_mechanism.txt')